# Simplicits Debug on Surgical Tissue Point Cloud (Gravity Only)

This notebook loads a **surface point cloud** from your pickle file and runs a **first-pass Simplicits simulation under gravity** (no tool constraints yet).

**Notes**
- This follows the structure of Kaolin's Simplicits Easy API example.
- If you run in **VS Code** and the Kaolin ipywidget visualizer fails, use the **k3d** visualization cells (they work in most environments).


In [1]:
# --- Imports ---
import os
import pickle
import numpy as np
import torch

import kaolin as kal

# For point cloud visualization (recommended for VS Code + Jupyter)
#   pip install k3d
import k3d

## 1) Load tissue point cloud from PKL

In [2]:
pkl_file_path = "/home/nan/Desktop/datasets/xpbd/StereoMIS_tissue_tool_trajectories_XPBD/sim_particles/tissue_pts_dnsampled_once.pkl"

with open(pkl_file_path, "rb") as f:
    data = pickle.load(f)

print(type(data), len(data))
print("keys:", data[0].keys())
print("xyz:", np.asarray(data[0]["xyz"]).shape)

<class 'list'> 200
keys: dict_keys(['frame_id', 'xyz', 'rgb', 'opacity', 'indices'])
xyz: (38426, 3)


## 2) Choose a rest frame and move to GPU

In [3]:
# Choose which frame to treat as rest state
rest_frame_idx = 0

xyz = np.asarray(data[rest_frame_idx]["xyz"], dtype=np.float32)  # (N,3)
N = xyz.shape[0]
print("N =", N)

pts = torch.from_numpy(xyz).cuda()

# Center + normalize into roughly [-1,1] cube (helps training stability)
pts = kal.ops.pointcloud.center_points(pts.unsqueeze(0), normalize=True).squeeze(0)

orig_pts = pts.clone()

N = 38426


## 3) Quick visualization of rest point cloud (k3d)

In [4]:
plot = k3d.plot()
k3d_pts = k3d.points(orig_pts.detach().cpu().numpy(), point_size=0.01)
plot += k3d_pts
plot.display()

Output()

## 4) Material fields (constant for now)

In [5]:
# These are *per-point* material fields used by the elastic loss / simulation.
# Start simple: constant fields.
# Units are not super important for this debug step; tune later.

yms  = torch.full((N,), 2e5, device=pts.device, dtype=pts.dtype)   # Young's modulus
prs  = torch.full((N,), 0.45, device=pts.device, dtype=pts.dtype)  # Poisson ratio
rhos = torch.full((N,), 1000., device=pts.device, dtype=pts.dtype) # Density

# Approx volume: for surface point clouds this is not "true volume".
# Use a rough proxy based on bounding box volume to get reasonable scaling.
mn = pts.min(dim=0).values
mx = pts.max(dim=0).values
bbox_vol = float(torch.prod(mx - mn).detach().cpu())
approx_volume = max(bbox_vol, 1e-6)

print("approx_volume (bbox proxy):", approx_volume)

approx_volume (bbox proxy): 0.6332738995552063


## 5) Train a SimplicitsObject (learn weights)

This step learns the **skinning weight field** \(w(x)\) (self-supervised) using elastic energy under random handle transforms.

Start with low iterations to debug the pipeline; increase later.


In [6]:
# Handles = reduced DOFs. Start small for debugging.
num_handles = 5

# Tip: increase training_num_steps after the first end-to-end run works.
sim_obj = kal.physics.simplicits.SimplicitsObject.create_trained(
    pts,
    yms,
    prs,
    rhos,
    approx_volume,
    num_handles=num_handles,

    training_num_steps=3000,
    training_lr_start=1e-3,
    training_lr_end=1e-3,

    # Coeffs similar to Kaolin example; adjust if training unstable.
    training_le_coeff=1e-1,
    training_lo_coeff=1e6,

    training_log_every=500,
    normalize_for_training=True,
)

print("trained object:", sim_obj)

trained object: <kaolin.physics.simplicits.easy_api.SimplicitsObject object at 0x7fab421e9060>


## 6) Build a scene and enable gravity-only simulation

In [47]:
scene = kal.physics.simplicits.SimplicitsScene()

# Debug-friendly settings
scene.max_newton_steps = 8
scene.timestep = 0.02
scene.direct_solve = True
floor_h = -1.5

obj_idx = scene.add_object(sim_obj)

# Kaolin convention: world_up_axis=1 => gravity along +Y in their example.
# If you want gravity downward, flip sign.
scene.set_scene_gravity(acc_gravity=torch.tensor([0.0, +9.8, 0.0], device=pts.device, dtype=pts.dtype))
# scene.set_scene_gravity(acc_gravity=torch.tensor([0.0, -9.8, 0.0], device=pts.device, dtype=pts.dtype))
scene.set_scene_floor(floor_height=floor_h, floor_axis=1, floor_penalty=1000)

# No floor yet (debug). Add later once motion looks sane.
# scene.set_scene_floor(floor_height=-0.8, floor_axis=1, floor_penalty=1000.0)

scene.reset_scene()

## 7) Run a few sim steps and update the point cloud in k3d

In [48]:
# Choose which frame to treat as rest state
rest_frame_idx = 0

xyz = np.asarray(data[rest_frame_idx]["xyz"], dtype=np.float32)  # (N,3)
N = xyz.shape[0]
print("N =", N)

pts = torch.from_numpy(xyz).cuda()

# Center + normalize into roughly [-1,1] cube (helps training stability)
pts = kal.ops.pointcloud.center_points(pts.unsqueeze(0), normalize=True).squeeze(0)

orig_pts = pts.clone()

N = 38426


In [50]:
def get_deformed_points():
    # Returns (N,3) torch tensor on GPU
    return scene.get_object_deformed_pts(obj_idx, orig_pts)

# Warm-up fetch
deformed = get_deformed_points()
k3d_pts.positions = deformed.detach().cpu().numpy()

min_y = float(orig_pts[:, 1].min().detach().cpu())
max_y = float(orig_pts[:, 1].max().detach().cpu())
print(f"init min_y={min_y:.4f} max_y={max_y:.4f}  (floor={floor_h})")

for s in range(100):
    scene.run_sim_step()
    if s % 2 == 0:
        deformed = get_deformed_points()
        k3d_pts.positions = deformed.detach().cpu().numpy()
    X = scene.get_object_deformed_pts(obj_idx, orig_pts)  # (N,3)
    min_y = float(X[:, 1].min().detach().cpu())
    max_y = float(X[:, 1].max().detach().cpu())
    if s % 10 == 0:
        print(f"step {s:04d}  min_y={min_y:.4f} max_y={max_y:.4f}  (floor={floor_h})")

init min_y=-0.3944 max_y=0.3944  (floor=-1.5)
step 0000  min_y=-0.3984 max_y=0.3905  (floor=-1.5)
step 0010  min_y=-0.6532 max_y=0.1357  (floor=-1.5)
step 0020  min_y=-1.2999 max_y=-0.5111  (floor=-1.5)
step 0030  min_y=-1.6175 max_y=-1.1608  (floor=-1.5)
step 0040  min_y=-1.5361 max_y=-0.9372  (floor=-1.5)
step 0050  min_y=-1.5989 max_y=-1.0561  (floor=-1.5)
step 0060  min_y=-1.5618 max_y=-1.1623  (floor=-1.5)
step 0070  min_y=-1.5519 max_y=-1.0842  (floor=-1.5)
step 0080  min_y=-1.5575 max_y=-1.1317  (floor=-1.5)
step 0090  min_y=-1.5557 max_y=-1.1247  (floor=-1.5)


In [49]:
plot = k3d.plot()
k3d_pts = k3d.points(orig_pts.detach().cpu().numpy(), point_size=0.01)
plot += k3d_pts
plot.display()

Output()

## 7b) Optional: Kaolin `IpyTurntableVisualizer` widget view (mesh-based)

Kaolin's `IpyTurntableVisualizer` renders **meshes** (triangles).  
Your tissue input is a **surface point cloud**, so we either:

1) Reconstruct a triangle mesh from the points (requires `open3d`), or  
2) Skip this section and keep using the `k3d` point visualization.

If you previously hit the ipywidget error inside VS Code, run this notebook in **JupyterLab / classic notebook**.


In [11]:
# --- Mesh reconstruction (optional) ---
# If you already have a tissue surface mesh .obj, you can just import it with kal.io.import_mesh(...).
# Otherwise, we can reconstruct a mesh from the point cloud using Open3D.

import copy
import threading
from ipywidgets import Button, HBox, VBox

try:
    import open3d as o3d
    _HAS_O3D = True
except Exception as e:
    _HAS_O3D = False
    print("[INFO] open3d not available; skipping mesh reconstruction.")
    print("       Install: pip install open3d  (in the same env as this notebook)")
    print("       Error:", repr(e))

mesh = None
orig_vertices = None

if _HAS_O3D:
    # Use the rest points as mesh vertices
    pts_cpu = orig_pts.detach().cpu().numpy().astype("float64")

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts_cpu)

    # Estimate normals (needed for Poisson / BPA). Tune radius if needed.
    pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=50))
    pcd.orient_normals_consistent_tangent_plane(50)

    # Poisson reconstruction (robust for noisy point clouds). Depth controls resolution.
    # If this is too slow, lower depth (e.g., 7 or 6).
    poisson_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=7)
    poisson_mesh.remove_duplicated_triangles()
    poisson_mesh.remove_degenerate_triangles()
    poisson_mesh.remove_non_manifold_edges()
    poisson_mesh.compute_vertex_normals()

    V = np.asarray(poisson_mesh.vertices, dtype=np.float32)
    F = np.asarray(poisson_mesh.triangles, dtype=np.int64)

    # (Optional) Crop low-density regions (often removes floaters)
    dens = np.asarray(densities)
    keep = dens > np.quantile(dens, 0.05)
    poisson_mesh = poisson_mesh.select_by_index(np.where(keep)[0])
    V = np.asarray(poisson_mesh.vertices, dtype=np.float32)
    F = np.asarray(poisson_mesh.triangles, dtype=np.int64)

    # Move to GPU as a Kaolin SurfaceMesh
    Vt = torch.from_numpy(V).to(device=orig_pts.device, dtype=orig_pts.dtype)
    Ft = torch.from_numpy(F).to(device=orig_pts.device)

    # IMPORTANT: We trained/created Simplicits on `orig_pts` (N points).
    # If Poisson creates a mesh with a *different* number of vertices than orig_pts,
    # we can't directly update mesh.vertices with scene.get_object_deformed_pts(obj_idx, orig_pts).
    # For the widget, we want mesh vertices == orig_pts, so we re-use orig_pts as vertices and
    # build faces from Poisson by nearest-neighbor remap.
    #
    # For debugging, we'll do a simpler route:
    #   - Use the *original points* as the mesh vertices (orig_pts)
    #   - Use a crude triangulation from Poisson faces by mapping to nearest orig point.
    #
    # This keeps vertex count consistent with the simulator.
    import scipy.spatial
    tree = scipy.spatial.cKDTree(orig_pts.detach().cpu().numpy())
    _, idx_map = tree.query(V, k=1)
    F_mapped = idx_map[F]  # triangles remapped to orig point indices

    Vt = orig_pts.clone()
    Ft = torch.from_numpy(F_mapped.astype(np.int64)).to(device=orig_pts.device)

    mesh = kal.rep.SurfaceMesh(vertices=Vt, faces=Ft)
    orig_vertices = mesh.vertices.clone()
    print("Reconstructed mesh for widget:", mesh)
else:
    print("[INFO] Widget section requires a triangle mesh. Keep using k3d, or install open3d.")

[INFO] open3d not available; skipping mesh reconstruction.
       Install: pip install open3d  (in the same env as this notebook)
       Error: ModuleNotFoundError("No module named 'open3d'")
[INFO] Widget section requires a triangle mesh. Keep using k3d, or install open3d.


In [12]:
# --- Widget-based renderer + sim buttons ---
# Only runs if `mesh` exists.

if mesh is not None:
    resolution = 512
    camera = kal.render.easy_render.default_camera(resolution).cuda()

    light_direction = kal.render.lighting.sg_direction_from_azimuth_elevation(1., 1.)
    lighting = kal.render.lighting.SgLightingParameters(
        amplitude=3., sharpness=5., direction=light_direction
    ).cuda()

    def render(in_cam):
        active_pass = kal.render.easy_render.RenderPass.render
        render_res = kal.render.easy_render.render_mesh(in_cam, mesh, lighting=lighting)

        img = render_res[active_pass]
        background_mask = (render_res[kal.render.easy_render.RenderPass.face_idx] < 0).bool()
        img2 = torch.clamp(img, 0, 1)[0]
        img2[background_mask[0]] = 1
        final = (img2 * 255.).to(torch.uint8)
        return {
            "img": final,
            "face_idx": render_res[kal.render.easy_render.RenderPass.face_idx].squeeze(0).unsqueeze(-1)
        }

    def fast_render(in_cam, factor=8):
        lowres_cam = copy.deepcopy(in_cam)
        lowres_cam.width = in_cam.width // factor
        lowres_cam.height = in_cam.height // factor
        return render(lowres_cam)

    # Reset mesh to its rest state
    mesh.vertices = orig_vertices

    global sim_thread_open, sim_thread
    sim_thread_open = False
    sim_thread = None

    def reset_simulation(visualizer):
        global scene
        with visualizer.out:
            scene.reset_scene()
        mesh.vertices = scene.get_object_deformed_pts(obj_idx, orig_vertices)
        visualizer.render_update()

    def run_sim(num_steps=200):
        for s in range(num_steps):
            with visualizer.out:
                scene.run_sim_step()
                if s % 10 == 0:
                    print(".", end="")
            mesh.vertices = scene.get_object_deformed_pts(obj_idx, orig_vertices)
            visualizer.render_update()

    def start_simulation(_):
        global sim_thread_open, sim_thread
        with visualizer.out:
            if sim_thread_open:
                sim_thread.join()
                sim_thread_open = False
            sim_thread_open = True
            sim_thread = threading.Thread(target=run_sim, daemon=True)
            sim_thread.start()

    visualizer = kal.visualize.IpyTurntableVisualizer(
        resolution, resolution, copy.deepcopy(camera), render, fast_render=fast_render,
        max_fps=24, world_up_axis=1
    )

    buttons = [Button(description=x) for x in ['Run Sim', 'Reset']]
    buttons[0].on_click(start_simulation)
    buttons[1].on_click(lambda _: reset_simulation(visualizer))

    reset_simulation(visualizer)
    display(HBox([visualizer.canvas, VBox(buttons)]), visualizer.out)
else:
    print("[INFO] Mesh is None -> skipping IpyTurntableVisualizer. Use the k3d view above.")

[INFO] Mesh is None -> skipping IpyTurntableVisualizer. Use the k3d view above.


## 8) Next: add a simple attachment constraint (preview)

Once gravity-only works, the simplest first constraint is a **soft attachment penalty** on a subset of points.

Conceptually:
\[
E_{attach}(z)=\frac{k}{2}\sum_{i\in\mathcal{I}}\|x_i(z)-x_i^*\|^2
\]

Implementation depends on whether you attach in:
- vertex/point space (easy if you can add an energy term), or
- handle space by modifying z directly (less physical).

If you paste the specific scene API you’re using to add boundary/penalty constraints (or show the functions in your `easy_api.py`), we can wire it in cleanly.
